#Example 11.2 Creating a Generative AI Legal  Solution

In [ ]:
# 1 Install Google Cloud SDK
# If you haven't installed the SDK: https://cloud.google.com/sdk/docs/install

# 2 Initialize GCP environment
gcloud init --skip-diagnostics

# 3 Create and Set Up Cloud Storage Bucket
gsutil mb -l us-central1 gs://my-legal-data-input-bucket/
gsutil mb -l us-central1 gs://my-legal-data-output-bucket/
gsutil mb -l us-central1 gs://finetuned_model/

# 4 Upload PDF Files to Cloud Storage
gsutil cp -r D:/0_LawGPT/0_Contract_Data/* gs://my-legal-data-bucket/

# 5 Set Up a Compute Engine VM with GPUs
gcloud compute instances create llama3-fine-tune-vm \
    --zone=us-central1-a \
    --machine-type=n1-standard-8 \
    --accelerator=type=nvidia-tesla-p4,count=1 \
    --boot-disk-size=200GB \
    --image-family=tf-latest-gpu \
    --image-project=deeplearning-platform-release \
    --maintenance-policy=TERMINATE

# 6 SSH into the VM
gcloud compute ssh llama3-fine-tune-vm --zone=us-central1-a

# 7 Install Required Packages on the VM
sudo apt update
sudo apt install python3-pip
pip3 install torch transformers datasets
pip3 install PyMuPDF

# 8 Convert PDFs into a Trainable Format
import fitz  # PyMuPDF
import os

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

def save_text(text, output_path):
    with open(output_path, "w") as f:
        f.write(text)

def convert_pdfs_to_texts(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for pdf_file in os.listdir(input_folder):
        if pdf_file.endswith(".pdf"):
            pdf_path = os.path.join(input_folder, pdf_file)
            text = extract_text_from_pdf(pdf_path)
            text_file = pdf_file.replace(".pdf", ".txt")
            output_path = os.path.join(output_folder, text_file)
            save_text(text, output_path)

input_folder = "gs://my-legal-data-input-bucket/"
output_folder = "gs://my-legal-data-output-bucket/"
convert_pdfs_to_texts(input_folder, output_folder)

# 9 Download the Pre-trained LLaMA3 Model
from transformers import LlamaForCausalLM, LlamaTokenizer

model_name = "huggingface/llama3-7B"  # Update this if required
tokenizer = LlamaTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(model_name)

# 10 Prepare Dataset for Fine-Tuning
from datasets import Dataset
import os

def load_texts_from_folder(folder_path):
    texts = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            with open(os.path.join(folder_path, file_name), "r") as f:
                texts.append(f.read())
    return texts

text_folder = "gs://my-legal-data-output-bucket/"
texts = load_texts_from_folder(text_folder)
dataset = Dataset.from_dict({"text": texts})

# 11 Tokenize the dataset:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 12 Fine-Tune the Model
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama3-finetuned-legal",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    evaluation_strategy="steps",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

# 13 Save the Fine-Tuned Model
trainer.save_model("gs://finetuned_model/")

# 14 Deploy the Model on AI Platform
gcloud ai models upload --region us-central1 \
    --display-name "llama3-finetuned-legal" \
    --artifact-uri gs://finetuned_model/

# 15 Create an endpoint for serving predictions.
gcloud ai endpoints create --region us-central1 \
    --display-name "llama3-finetuned-endpoint"

# 16 Make Predictions
gcloud ai endpoints predict --region us-central1 \
    --endpoint-id [ENDPOINT_ID] \
    --json-request '[{"input": "what is section 402 of indian penal code"}]'


#Example 11.3 Creating a Generative AI Medical Solution

In [ ]:
# 1 Create a Google Cloud Project
# If you don't already have a GCP project, create one using the Google Cloud Console and note down your project ID.

# 2 Install Google Cloud SDK
# Install the SDK by following the instructions in the Google Cloud SDK Installation Guide and authenticate:
gcloud init --skip-diagnostics

# 3 Enable Required APIs
gcloud services enable compute.googleapis.com
gcloud services enable aiplatform.googleapis.com
gcloud services enable storage.googleapis.com

# 4 Set Up Google Cloud Storage (GCS) Bucket
gsutil mb -l us-central1 gs://my-data-bucket/

# 5 Upload your raw PDF files (if needed)
gsutil cp /local/path/to/legal_data.pdf gs://my-data-bucket/raw_data/

# 6 Set Up AI Platform Notebook (for Preprocessing and Script Development)
# Go to Vertex AI > Workbench > Notebooks in the Google Cloud Console, create a new JupyterLab instance and open the notebook.

# 7 Convert PDF to Text (Using Cloud Functions)
import fitz  # PyMuPDF
from google.cloud import storage

def extract_text_from_pdf(event, context):
    bucket_name = event['bucket']
    file_name = event['name']

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(file_name)

    temp_file = f'/tmp/{file_name}'
    blob.download_to_filename(temp_file)

    doc = fitz.open(temp_file)
    extracted_text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        extracted_text += page.get_text()

    text_blob = bucket.blob(file_name.replace(".pdf", ".txt"))
    text_blob.upload_from_string(extracted_text)

# Deploy the Cloud Function:
gcloud functions deploy pdf_to_text --runtime python39 --trigger-resource gs://my-data-bucket/raw_data/ --trigger-event google.storage.object.finalize

# 8 Preprocess Text for Model Training
!pip install nltk spacy
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

from google.cloud import storage

def load_text_from_gcs(bucket_name, file_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)
    return blob.download_as_string().decode('utf-8')

text_data = load_text_from_gcs('your_bucket_name', 'path_to_data/legal_data.txt')
cleaned_text = preprocess_text(text_data)

def save_to_gcs(data, bucket_name, file_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)
    blob.upload_from_string(data)

save_to_gcs(cleaned_text, 'your_bucket_name', 'path_to_data/cleaned_legal_data.txt')

# 9 Install transformers and other dependencies
!pip install transformers datasets

# 10 Fine-tune the LLaMA3 7B model:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("meta-llama/LLaMA-3B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/LLaMA-3B")
dataset = load_dataset('text', data_files='gs://your_bucket_name/path_to_data/cleaned_legal_data.txt')

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['train']
)

trainer.train()

trainer.save_model("gs://your_bucket_name/model_output/")

# 11 Submit a Training Job to Vertex AI
gcloud ai custom-jobs create --region=us-central1 \
    --display-name=llama3-finetune \
    --python-package-uris=gs://your_bucket_name/path_to_package/your_package.tar.gz \
    --python-module=your_package_name.finetune_script \
    --job-dir=gs://your_bucket_name/job_output/ \
    --replica-count=1 \
    --machine-type=n1-standard-8 \
    --accelerator=type=NVIDIA_TESLA_V100,count=1 \
    --args="--train_data=gs://your_bucket_name/path_to_data/cleaned_legal_data.txt", "--output_dir=gs://your_bucket_name/model_output/"

# 12 Upload and Deploy the Model
gcloud ai models upload --region=us-central1 \
    --display-name=llama3-finetuned \
    --artifact-uri=gs://your_bucket_name/model_output/ \
    --container-image-uri=us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-8:latest

# 13 Make Predictions
gcloud ai endpoints create --region=us-central1 --display-name=llama3-finetuned-endpoint

gcloud ai endpoints deploy-model ENDPOINT_ID --region=us-central1 --model=MODEL_ID --display-name=llama3-finetuned --machine-type=n1-standard-4

gcloud ai endpoints predict --region=us-central1 --endpoint=ENDPOINT_ID --json-request='{"instances": ["Example input text"]}'


#Example 11.4 Creating a Customer Care Digital Avatar Using NVIDIA Maxine on GCP

In [ ]:
# 1 Create a Google Cloud Project
# Go to the Google Cloud Console, create a new project, and note down the Project ID.

# 2 Install Google Cloud SDK and Authenticate
# Install the SDK by following the Google Cloud SDK Installation Guide.
gcloud init

# 3 Enable Required APIs
gcloud services enable compute.googleapis.com
gcloud services enable aiplatform.googleapis.com
gcloud services enable storage.googleapis.com

# 4 Set Up Google Cloud Storage
gsutil mb -l us-central1 gs://my-customer-avatar-bucket/

# 5 Set Up a Compute Engine VM with GPU
gcloud compute instances create maxine-customer-avatar-vm \
--zone=us-central1-a \
--machine-type=n1-standard-8 \
--accelerator=type=nvidia-tesla-p4,count=1 \
--boot-disk-size=200GB \
--image-family=tf-latest-gpu \
--image-project=deeplearning-platform-release \
--maintenance-policy=TERMINATE

# 6 Install NVIDIA Drivers and SDKs
gcloud compute ssh maxine-customer-avatar-vm --zone=us-central1-a

# 7 Install NVIDIA drivers and CUDA toolkit
sudo apt update
sudo apt install nvidia-driver-470
sudo apt install cuda-toolkit-11-2

# 8 Install NVIDIA Maxine SDK
# Follow NVIDIA Maxine SDK installation guide and verify the installation:
nvmaxine --version

# 9 Use NVIDIA Maxine Audio Effects
import nvidia.maxine.audio as nv_audio
audio_engine = nv_audio.AudioEffects()

def process_audio(input_audio):
    enhanced_audio = audio_engine.noise_suppression(input_audio)
    enhanced_audio = audio_engine.speech_enhancement(enhanced_audio)
    return enhanced_audio

# 10 Use NVIDIA Maxine Face Tracking and Animation
import nvidia.maxine.face as nv_face
face_engine = nv_face.FaceTracker()
face_animation = nv_face.FaceAnimation()

def animate_avatar(webcam_frame):
    face_landmarks = face_engine.track_face(webcam_frame)
    animated_frame = face_animation.apply_animation(face_landmarks)
    return animated_frame

# 11 Integrate Text-to-Speech (TTS) with Maxine
from google.cloud import texttospeech

tts_client = texttospeech.TextToSpeechClient()

def synthesize_speech(text):
    input_text = texttospeech.SynthesisInput(text=text)
    voice = texttospeech.VoiceSelectionParams(language_code="en-US", ssml_gender=texttospeech.SsmlVoiceGender.NEUTRAL)
    audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)
    response = tts_client.synthesize_speech(input=input_text, voice=voice, audio_config=audio_config)
    return response.audio_content

# 12 Deploy Avatar Backend on GCP (Flask-based backend)
from flask import Flask, request, jsonify
import nvidia.maxine.audio as nv_audio
import nvidia.maxine.face as nv_face
from google.cloud import texttospeech

app = Flask(__name__)

@app.route("/process_avatar", methods=["POST"])
def process_avatar():
    input_data = request.json
    processed_audio = process_audio(input_data['audio'])
    animated_avatar = animate_avatar(input_data['video_frame'])
    return jsonify({"audio": processed_audio, "avatar_frame": animated_avatar})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)

# 13 Build and Deploy the container to Google Cloud Run
gcloud builds submit --tag gcr.io/[PROJECT_ID]/customer-avatar-service
gcloud run deploy customer-avatar-service --image gcr.io/[PROJECT_ID]/customer-avatar-service --platform managed --allow-unauthenticated


#Example 11.5 Streamlining Drug Discovery with Generative Virtual Screening Using NVIDIA NIM Agent Blueprint on GCP

In [ ]:
# 1 Create a GCP project
gcloud projects create my-drug-discovery-project --set-as-default

# 2 Enable the required APIs
gcloud services enable compute.googleapis.com
gcloud services enable aiplatform.googleapis.com
gcloud services enable storage.googleapis.com

# 3 Set up a Compute Engine VM with NVIDIA GPU support
gcloud compute instances create gpu-instance --zone=us-central1-a \
    --machine-type=n1-standard-8 \
    --accelerator=type=nvidia-tesla-v100,count=1 \
    --image-family=common-cu110 \
    --image-project=nvidia-ngc-public

# 4 Install NVIDIA NGC CLI and authenticate
curl -O https://ngc.nvidia.com/downloads/ngccli_cat_linux.zip && unzip ngccli_cat_linux.zip
chmod u+x ngc-cli/ngc
./ngc-cli/ngc config set

# 5 Prepare Data for Virtual Screening
# Use RDKit to process molecular data in SMILES format
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load and preprocess SMILES dataset
smiles_list = ['CCO', 'CCN', 'CCC']
molecules = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# Extract molecular descriptors
for mol in molecules:
    mw = Descriptors.MolWt(mol)  # Molecular weight
    logp = Descriptors.MolLogP(mol)  # LogP
    print(f'MW: {mw}, LogP: {logp}')

# 6 Configure Generative Virtual Screening NIM Agent
pip install rdkit pytorch-lightning

# Download the Generative Virtual Screening NIM Agent Blueprint
ngc registry resource download-version nvidia/nim-gvs-agent:latest

# 7 Generate Drug Candidates
from transformers import GPT2LMHeadModel, GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('nvidia/gvs-molecule-generator')
model = GPT2LMHeadModel.from_pretrained('nvidia/gvs-molecule-generator')

# Generate molecular candidates
input_text = "CCN"
input_ids = tokenizer.encode(input_text, return_tensors='pt')
output = model.generate(input_ids, max_length=50, num_return_sequences=10)
generated_molecules = [tokenizer.decode(output[i], skip_special_tokens=True) for i in range(10)]
print(generated_molecules)

# 8 Deploy and Scale Solution
# Package and upload the model to GCP storage
gcloud storage cp gvs-model.tar.gz gs://my-bucket/gvs-model/

# Submit a training job to Vertex AI
gcloud ai custom-jobs create --region=us-central1 --display-name=gvs-drug-discovery \
    --python-package-uris=gs://my-bucket/gvs-model.tar.gz \
    --python-module=my_gvs_model.train \
    --machine-type=n1-standard-8 --accelerator=type=NVIDIA_TESLA_V100,count=1

# Scale the GPU resources
gcloud compute instances set-machine-type gpu-instance --machine-type=n1-standard-16 --accelerator=type=nvidia-tesla-v100,count=4
